# How do you save and deploy models?
**Topics:** SavedModel Format · model.save() · load_model() · TFLite Conversion · TF Serving Basics

---
## 1. Saving Models

### Two formats

| Format | Extension | Use |
|---|---|---|
| SavedModel | directory or `.keras` | Production, TF Serving, recommended |
| HDF5 | `.h5` | Legacy, single file |

### What gets saved
- Model architecture
- Weights and biases
- Optimizer state (for resuming training)
- Training configuration (from `compile()`)

### Key intuition
- SavedModel is a directory containing the computation graph and weights
- `.keras` format (new in TF 2.12+) is the recommended modern format
- HDF5 is a single binary file — easier to move around but less flexible

### Gotchas
- Subclassed models need `save_traces=True` in SavedModel format
- `save_weights_only=True` in ModelCheckpoint saves faster but needs architecture to load
- Always test loading before deploying — verify outputs match

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import os

np.random.seed(42); tf.random.set_seed(42)

X = np.random.randn(200, 8).astype(np.float32)
y = (X[:, 0] + X[:, 1] > 0).astype(np.int32)

model = tf.keras.Sequential([
    tf.keras.layers.Dense(32, activation='relu', input_shape=(8,)),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(1,  activation='sigmoid'),
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(X[:160], y[:160], epochs=10, verbose=0)

# --- Save in different formats (Keras 3 / TF 2.16+) ---
# model.save() only supports .keras and .h5 in Keras 3
model.save('/tmp/model.keras')            # Recommended native Keras format
model.save('/tmp/model.h5')              # Legacy HDF5 format

# For SavedModel/TFLite/TFServing: use model.export() instead
model.export('/tmp/model_savedmodel')    # Exports TF SavedModel for serving

print("Saved files:")
for path in ['/tmp/model.keras', '/tmp/model.h5', '/tmp/model_savedmodel']:
    if os.path.isdir(path):
        size = sum(os.path.getsize(os.path.join(r,f)) for r,d,files in os.walk(path) for f in files)
        print(f"  {path} (dir)  : {size/1024:.1f} KB")
    elif os.path.exists(path):
        print(f"  {path}: {os.path.getsize(path)/1024:.1f} KB")

# --- Loading ---
model_k   = tf.keras.models.load_model('/tmp/model.keras')
model_h5  = tf.keras.models.load_model('/tmp/model.h5')

X_test   = X[160:]
out_orig = model.predict(X_test, verbose=0)
out_k    = model_k.predict(X_test, verbose=0)
print(f"Outputs match (.keras): {np.allclose(out_orig, out_k, atol=1e-5)}")

# Visualize SavedModel directory structure
fig, ax = plt.subplots(figsize=(10, 5))
ax.set_xlim(0, 10); ax.set_ylim(0, 7); ax.axis('off')
ax.set_title('SavedModel Directory Structure', fontsize=13, fontweight='bold')

import matplotlib.patches as mpatches
def box(ax, x, y, w, h, label, color, fontsize=9):
    rect = mpatches.FancyBboxPatch((x, y), w, h,
        boxstyle='round,pad=0.07', facecolor=color, edgecolor='white', linewidth=1.5, alpha=0.9)
    ax.add_patch(rect)
    ax.text(x+w/2, y+h/2, label, ha='center', va='center',
            fontsize=fontsize, fontweight='bold', color='white')

box(ax, 3.5, 5.5, 3, 0.8, 'model_savedmodel/', '#2C3E50', 11)
box(ax, 1.0, 3.8, 2.5, 0.7, 'saved_model.pb', '#2980B9')
box(ax, 4.0, 3.8, 2.5, 0.7, 'variables/', '#8E44AD')
box(ax, 7.0, 3.8, 2.5, 0.7, 'assets/', '#27AE60')
box(ax, 3.5, 2.2, 3.0, 0.7, 'variables.data-00000-of-00001', '#8E44AD')
box(ax, 3.5, 1.2, 3.0, 0.7, 'variables.index', '#8E44AD')

for src_xy, dst_xy in [((5.0,5.5),(2.25,4.5)), ((5.0,5.5),(5.25,4.5)), ((5.0,5.5),(8.25,4.5))]:
    ax.annotate('', xy=(dst_xy[0], dst_xy[1]), xytext=(src_xy[0], src_xy[1]-0.0),
                arrowprops=dict(arrowstyle='->', color='#555', lw=1.5))
for src_xy, dst_xy in [((5.25,3.8),(5.0,2.9)), ((5.25,3.8),(5.0,1.9))]:
    ax.annotate('', xy=(dst_xy[0], dst_xy[1]), xytext=(src_xy[0], src_xy[1]),
                arrowprops=dict(arrowstyle='->', color='#555', lw=1.5))

labels = [('2.3, 4.0', 'Computation graph (architecture)'),
          ('7.0, 4.0', 'Optional asset files'),
          ('1.2, 3.5', 'Weights (binary)'),
          ('1.2, 2.5', 'Weight index')]
for pos, note in [((2.25, 3.2), 'Computation graph (architecture)'),
                  ((8.25, 3.2), 'Optional asset files')]:
    ax.text(pos[0], pos[1], note, ha='center', fontsize=7.5, color='#555', style='italic')

plt.tight_layout()
plt.savefig('savedmodel_structure.png', dpi=120, bbox_inches='tight')
plt.show()


---
## 2. TFLite Conversion

### What it is
- TensorFlow Lite — lightweight format for mobile and edge deployment
- Converts SavedModel to a single `.tflite` file
- Supports quantization — reduces model size and speeds up inference

### Quantization types

| Type | Size reduction | Accuracy loss | Use case |
|---|---|---|---|
| None (float32) | 1x | None | Server inference |
| Dynamic range | ~4x | Minimal | Mobile, default choice |
| Float16 | ~2x | Minimal | GPU mobile |
| Integer (int8) | ~4x | Small | Microcontrollers, edge |

### Key intuition
- Quantization replaces float32 weights with int8 — 4x smaller, faster on hardware
- Dynamic range quantization — no calibration data needed — easiest to apply
- Full int8 quantization — needs representative dataset — most aggressive

### Gotchas
- TFLite does not support all TF ops — check op compatibility before converting
- Quantization can hurt accuracy for small models — always benchmark
- TFLite interpreter API is different from Keras model API

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import os

np.random.seed(42)
X = np.random.randn(200, 8).astype(np.float32)
y = (X[:, 0] + X[:, 1] > 0).astype(np.int32)

model = tf.keras.Sequential([
    tf.keras.layers.Dense(64, activation='relu', input_shape=(8,)),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1,  activation='sigmoid'),
])
model.compile(optimizer='adam', loss='binary_crossentropy')
model.fit(X[:160], y[:160], epochs=10, verbose=0)
model.export('/tmp/model_for_tflite')   # Use export() for SavedModel in Keras 3

# Convert to TFLite (no quantization)
converter = tf.lite.TFLiteConverter.from_saved_model('/tmp/model_for_tflite')
tflite_model = converter.convert()
with open('/tmp/model.tflite', 'wb') as f:
    f.write(tflite_model)

# Convert with dynamic range quantization
converter_q = tf.lite.TFLiteConverter.from_saved_model('/tmp/model_for_tflite')
converter_q.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_quant = converter_q.convert()
with open('/tmp/model_quant.tflite', 'wb') as f:
    f.write(tflite_quant)

orig_size  = sum(os.path.getsize(os.path.join(r,f))
                  for r,d,files in os.walk('/tmp/model_for_tflite') for f in files) / 1024
lite_size  = os.path.getsize('/tmp/model.tflite')  / 1024
quant_size = os.path.getsize('/tmp/model_quant.tflite') / 1024

print(f"SavedModel size    : {orig_size:.1f} KB")
print(f"TFLite size        : {lite_size:.1f} KB")
print(f"TFLite quant size  : {quant_size:.1f} KB")

# TFLite inference
interpreter = tf.lite.Interpreter(model_path='/tmp/model.tflite')
interpreter.allocate_tensors()
inp_details = interpreter.get_input_details()
out_details = interpreter.get_output_details()
print(f"Input  shape: {inp_details[0]['shape']}")
print(f"Output shape: {out_details[0]['shape']}")

x_test = X[160:161]
interpreter.set_tensor(inp_details[0]['index'], x_test)
interpreter.invoke()
tflite_out = interpreter.get_tensor(out_details[0]['index'])
keras_out  = model.predict(x_test, verbose=0)
print(f"Keras output : {keras_out[0][0]:.4f}")
print(f"TFLite output: {tflite_out[0][0]:.4f}")

# Size and latency comparison
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
formats = ['SavedModel (float32)', 'TFLite (float32)', 'TFLite (quantized)']
sizes   = [orig_size, lite_size, quant_size]
colors  = ['#3498DB', '#F39C12', '#2ECC71']

bars = axes[0].bar(formats, sizes, color=colors, alpha=0.85, edgecolor='white', width=0.5)
axes[0].set_ylabel('File size (KB)', fontsize=11)
axes[0].set_title('Model Size Comparison', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='y')
axes[0].spines['top'].set_visible(False); axes[0].spines['right'].set_visible(False)
for bar, val in zip(bars, sizes):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                 f'{val:.1f} KB', ha='center', va='bottom', fontsize=10, fontweight='bold')

import time
n_runs = 100
keras_times, tflite_times = [], []
for _ in range(n_runs):
    t = time.time(); model.predict(X[160:161], verbose=0); keras_times.append(time.time()-t)
for _ in range(n_runs):
    t = time.time()
    interpreter.set_tensor(inp_details[0]['index'], X[160:161])
    interpreter.invoke()
    tflite_times.append(time.time()-t)

axes[1].boxplot([keras_times, tflite_times], labels=['Keras predict', 'TFLite invoke'],
                patch_artist=True,
                boxprops=dict(facecolor='#3498DB', alpha=0.7),
                medianprops=dict(color='white', linewidth=2))
axes[1].set_ylabel('Inference time (s)', fontsize=11)
axes[1].set_title(f'Inference Latency ({n_runs} runs)', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')
axes[1].spines['top'].set_visible(False); axes[1].spines['right'].set_visible(False)

plt.suptitle('TFLite: Size and Latency', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('tflite_comparison.png', dpi=120, bbox_inches='tight')
plt.show()


---
## 3. TF Serving Basics

### What it is
- Production serving system for TensorFlow models
- Exposes REST and gRPC endpoints for model inference
- Supports versioning, A/B testing, model hot-swapping

### Key intuition
- Save model in SavedModel format → TF Serving loads it automatically
- Send POST requests to `http://localhost:8501/v1/models/model_name:predict`
- Scales horizontally — multiple servers can serve the same model

### Interview Q&A

**What is the difference between TFLite and TF Serving?**
- TFLite → edge/mobile deployment (phones, microcontrollers, offline)
- TF Serving → server-side production deployment with REST/gRPC API

**How do you version models in TF Serving?**
- Save models in numbered subdirectories: `model/1/`, `model/2/`
- TF Serving automatically serves the latest version
- Specify version in URL: `/v1/models/model_name/versions/2:predict`

### Gotchas
- TF Serving requires SavedModel format — not `.h5` or `.tflite`
- Input must match model's input signature exactly — shape and dtype
- Batch size must be specified or use dynamic batch size (`-1`)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# TF Serving pattern reference
print("=== TF Serving Deployment Pattern ===")
print()
print("1. Save model:")
print("   model.save('/models/my_model/1/')  # version 1")
print()
print("2. Start TF Serving (Docker):")
print("   docker run -p 8501:8501 -p 8500:8500 \\")
print("     -v /models/my_model:/models/my_model \\")
print("     -e MODEL_NAME=my_model \\")
print("     tensorflow/serving")
print()
print("3. Send REST request:")
print("")
print("import requests, json")
print("data = {'instances': X_test.tolist()}")
print("response = requests.post(")
print("    'http://localhost:8501/v1/models/my_model:predict',")
print("    json=data")
print(")")
print("predictions = response.json()['predictions']")
print("")
print("4. Check model status:")
print("   GET http://localhost:8501/v1/models/my_model")

# Deployment architecture diagram
fig, ax = plt.subplots(figsize=(12, 6))
ax.set_xlim(0, 13); ax.set_ylim(0, 7); ax.axis('off')
ax.set_title('TF Model Deployment Architecture', fontsize=13, fontweight='bold')

def box(ax, x, y, w, h, label, color, fontsize=9):
    rect = mpatches.FancyBboxPatch((x, y), w, h,
        boxstyle='round,pad=0.07', facecolor=color, edgecolor='white', linewidth=1.5, alpha=0.9)
    ax.add_patch(rect)
    ax.text(x+w/2, y+h/2, label, ha='center', va='center',
            fontsize=fontsize, fontweight='bold', color='white')

def arr(ax, x1, y1, x2, y2, label='', color='#555555'):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color=color, lw=2))
    if label:
        mx, my = (x1+x2)/2, (y1+y2)/2
        ax.text(mx, my+0.15, label, ha='center', fontsize=8, color=color)

box(ax, 0.3, 5.5, 2.5, 0.8, 'Training (Keras model.fit)', '#2980B9')
box(ax, 0.3, 3.8, 2.5, 0.8, 'SavedModel (/models/v1/)', '#8E44AD')
box(ax, 3.5, 3.8, 2.5, 0.8, 'TF Serving (REST:8501)', '#E74C3C')
box(ax, 6.7, 5.2, 2.5, 0.8, 'Web App (client)', '#27AE60')
box(ax, 6.7, 3.8, 2.5, 0.8, 'Mobile App (TFLite)', '#F39C12')
box(ax, 6.7, 2.4, 2.5, 0.8, 'Edge Device (TFLite/int8)', '#1ABC9C')
box(ax, 9.9, 3.8, 2.5, 0.8, 'Response (predictions)', '#7F8C8D')

arr(ax, 1.55, 5.5, 1.55, 4.6, 'save()')
arr(ax, 2.8, 4.2, 3.5, 4.2, 'deploy')
arr(ax, 6.0, 4.2, 6.7, 5.6, 'POST /predict')
arr(ax, 6.0, 4.2, 6.7, 4.2, 'load model')
arr(ax, 6.0, 4.2, 6.7, 2.8, 'TFLite convert')
arr(ax, 9.2, 5.6, 9.9, 4.6, 'JSON resp')
arr(ax, 9.2, 4.2, 9.9, 4.2, '')

plt.tight_layout()
plt.savefig('tf_serving.png', dpi=120, bbox_inches='tight')
plt.show()


### Gotchas for all deployment
- Test the saved model outputs match training model outputs before deploying
- Version your models — always keep previous version available for rollback
- Monitor model performance in production — data drift can degrade accuracy over time
- For REST API, input JSON format: `{'instances': [[x1, x2, ...], [x1, x2, ...]]}` — list of lists